In [1]:
import os
import torch
import joblib
import numpy as np

# Path to your folder
data_dir = "/x1/ljiao/Bhanu/VIBE/cAvatar/dance"

# List all .pkl files
pkl_files = [f for f in os.listdir(data_dir) if f.endswith(".pkl")]
print(f"Found {len(pkl_files)} .pkl files.")

Found 425 .pkl files.


In [2]:
# Pick the first .pkl file and build full path
file_name = pkl_files[0]
file_path = os.path.join(data_dir, file_name)

# Load using joblib
data = joblib.load(file_path)

# Print all top-level keys and shapes
print(f"\n📂 Loaded file: {file_name}")
print("🔑 Top-level keys and their shapes:\n")

for key, value in data.items():
    if isinstance(value, np.ndarray):
        print(f"✅ {key}: {value.shape} (dtype: {value.dtype})")
    else:
        print(f"ℹ️  {key}: {type(value)}")


📂 Loaded file: G_I_DLE_G_I_DLE_Senorita_Senorita_Dance_Cover_Cover_by_Sol_E_Mirror_Mode_gFO_Fmy5aGEkpop_dance_12_clip1.pkl
🔑 Top-level keys and their shapes:

ℹ️  1: <class 'dict'>


In [3]:
# Safely inspect all keys first
print("Top-level keys:", data.keys())

# Now access using the correct key type
inner_data = data[1]

# Print the keys and shapes/types inside the nested dict
print("\n🔑 Keys inside data[1] and their shapes/types:\n")
for key, value in inner_data.items():
    if isinstance(value, np.ndarray):
        print(f"✅ {key}: {value.shape} (dtype: {value.dtype})")
    else:
        print(f"ℹ️  {key}: {type(value)}")


Top-level keys: dict_keys([1])

🔑 Keys inside data[1] and their shapes/types:

✅ pred_cam: (320, 3) (dtype: float32)
✅ orig_cam: (320, 4) (dtype: float64)
✅ verts: (320, 6890, 3) (dtype: float32)
✅ pose: (320, 72) (dtype: float32)
✅ betas: (320, 10) (dtype: float32)
✅ joints3d: (320, 49, 3) (dtype: float32)
✅ joints2d: (320, 49, 2) (dtype: float32)
✅ joints2d_img_coord: (320, 49, 2) (dtype: float32)
✅ bboxes: (320, 4) (dtype: float64)
✅ frame_ids: (320,) (dtype: int64)


*Original Code*

In [4]:
import os
import joblib
import numpy as np
from tqdm import tqdm

In [5]:
# Paths
pkl_dir = '/x1/ljiao/Bhanu/VIBE/cAvatar/dance'
frames_root = '/x1/ljiao/Bhanu/VIBE/output/dance'

# List all .pkl files
pkl_files = [f for f in os.listdir(pkl_dir) if f.endswith('.pkl')]

# Final dataset list
dataset = []

In [6]:
for file_name in tqdm(pkl_files, desc="🧠 Building dataset"):
    pkl_path = os.path.join(pkl_dir, file_name)
    base_name = file_name.replace('.pkl', '')
    frame_folder = os.path.join(frames_root, base_name + '_frames')

    if not os.path.isdir(frame_folder):
        print(f"⚠️ Skipping {file_name}: no matching frame folder found.")
        continue

    try:
        data = joblib.load(pkl_path)
        data = data[1]  # Single person assumption
        
    except Exception as e:
        print(f"❌ Failed to load {file_name}: {e}")
        continue

    frame_ids = data['frame_ids']
    for idx, frame_id in enumerate(frame_ids):
        # Frame name uses 1-based index, zero-padded
        frame_filename = f"{base_name}_{frame_id + 1:06d}.png"
        image_path = os.path.join(frame_folder, frame_filename)

        if not os.path.isfile(image_path):
            print(f"⚠️ Missing image: {image_path}")
            continue

        # Collect data for this frame
        frame_data = {
            'image_path': image_path,
            'pose': data['pose'][idx],          # (72,)
            'betas': data['betas'][idx],        # (10,)
            'cam': data['pred_cam'][idx],       # (3,)
            'joints2d': data['joints2d'][idx],  # (49, 2)
            'joints3d': data['joints3d'][idx],  # (49, 3)
            'verts': data['verts'][idx],        # (6890, 3)
            'video_name': file_name,
            'frame_id': int(frame_id)
        }

        dataset.append(frame_data)

print(f"\n✅ Dataset created with {len(dataset)} samples from {len(pkl_files)} videos.")


🧠 Building dataset:  16%|█▌        | 68/425 [00:00<00:02, 127.30it/s]

❌ Failed to load Small_short_skirt_of_rabbit_ears_COS_sister_curtilage_dance_3_jumpers_dance_26_clip1.pkl: 1


🧠 Building dataset:  42%|████▏     | 179/425 [00:01<00:01, 133.06it/s]

❌ Failed to load Ziqi_flip_jump_jump_gold_in_the_summer_please_return_to_Snapping_power_P2_vertical_screen_version_of_P1_golden_summer_Snapping_please_turn_to_jump_dance_7_clip1.pkl: 1


🧠 Building dataset: 100%|██████████| 425/425 [00:05<00:00, 76.23it/s] 


✅ Dataset created with 104816 samples from 425 videos.


In [7]:
# Print a few full records from the dataset (let's say first 3)
for i in range(1):
    sample = dataset[i]
    print(f"\n📦 Sample {i+1}")
    print("🖼️  Image Path:", sample['image_path'])
    print("🎥  Video:", sample['video_name'])
    print("🧠 Frame ID:", sample['frame_id'])
    
    print("\n🦴 Pose (72):")
    print(sample['pose'])

    print("\n📏 Betas (10):")
    print(sample['betas'])

    print("\n📷 Camera Params (3):")
    print(sample['cam'])

    print("\n📌 Joints 2D (49, 2):")
    print(sample['joints2d'])

    print("\n📌 Joints 3D (49, 3):")
    print(sample['joints3d'])

    print("\n🕸️ Verts (6890, 3): [showing only 5 sample vertices]")
    print(sample['verts'][:5])  # To keep output readable




📦 Sample 1
🖼️  Image Path: /x1/ljiao/Bhanu/VIBE/output/dance/G_I_DLE_G_I_DLE_Senorita_Senorita_Dance_Cover_Cover_by_Sol_E_Mirror_Mode_gFO_Fmy5aGEkpop_dance_12_clip1_frames/G_I_DLE_G_I_DLE_Senorita_Senorita_Dance_Cover_Cover_by_Sol_E_Mirror_Mode_gFO_Fmy5aGEkpop_dance_12_clip1_000001.png
🎥  Video: G_I_DLE_G_I_DLE_Senorita_Senorita_Dance_Cover_Cover_by_Sol_E_Mirror_Mode_gFO_Fmy5aGEkpop_dance_12_clip1.pkl
🧠 Frame ID: 0

🦴 Pose (72):
[-3.1366060e+00  3.9647389e-02 -9.6230723e-02 -1.3883905e-01
 -1.4649391e-02 -2.4378754e-02 -2.1023488e-01  5.9741706e-02
  5.7468362e-02  2.4631886e-01 -7.9255663e-03 -3.4149077e-02
  2.8004175e-01 -4.4994246e-02 -5.1459115e-02  4.3662962e-01
 -4.3419618e-02 -5.1518581e-03  5.2290760e-02 -6.8296224e-02
 -4.9057305e-02 -1.2119994e-01  1.5542796e-01 -4.4804569e-02
 -1.1574813e-01 -1.0524789e-01  6.1666984e-02 -3.4867402e-02
 -8.3612381e-03 -7.7008274e-03 -4.3903939e-02  2.9008523e-02
 -9.2144366e-03 -5.2485235e-02  9.6340757e-03  2.5114272e-02
 -1.5438434e-01  

In [8]:
import mediapipe as mp
import cv2
import numpy as np
from tqdm import tqdm

# Initialize MediaPipe pose detector
mp_pose = mp.solutions.pose
pose_detector = mp_pose.Pose(static_image_mode=True)

# Track failed cases
failed_samples = []

# Process each image in your dataset
for sample in tqdm(dataset, desc="🧠 Extracting MediaPipe 2D keypoints"):
    image_path = sample['image_path']
    
    try:
        # Load image
        image = cv2.imread(image_path)
        h, w, _ = image.shape
        image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        # Run MediaPipe
        results = pose_detector.process(image_rgb)

        # Extract 33 keypoints
        if results.pose_landmarks:
            keypoints = []
            for lm in results.pose_landmarks.landmark:
                x = lm.x  # already normalized
                y = lm.y  # already normalized
                keypoints.append([x, y])
            
            sample['mediapipe_kpts'] = np.array(keypoints)  # (33, 2)
        else:
            sample['mediapipe_kpts'] = None
            failed_samples.append(image_path)
    
    except Exception as e:
        print(f"❌ Error on image {image_path}: {e}")
        sample['mediapipe_kpts'] = None
        failed_samples.append(image_path)

print(f"\n✅ Keypoint extraction completed. Total samples: {len(dataset)}")
print(f"❌ Failed samples (no keypoints found): {len(failed_samples)}")


I0000 00:00:1743747110.395013 3032688 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1743747110.436731 3032865 gl_context.cc:369] GL version: 3.2 (OpenGL ES 3.2 NVIDIA 535.183.01), renderer: NVIDIA A100 80GB PCIe/PCIe/SSE2
🧠 Extracting MediaPipe 2D keypoints:   0%|          | 0/104816 [00:00<?, ?it/s]INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1743747110.521689 3032798 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1743747110.571813 3032821 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1743747110.587104 3032817 landmark_projection_calculator.cc:186] Using NORM_RECT without IMAGE_DIMENSIONS is only supported for the square ROI. Provide IMAGE_DIMENSIONS or use PROJECTION_MATRIX.
🧠 Extracting MediaPipe 2D keypoints


✅ Keypoint extraction completed. Total samples: 104816
❌ Failed samples (no keypoints found): 251


In [24]:
train_data = [s for s in dataset if s['mediapipe_kpts'] is not None]


In [25]:
sample = dataset[591]

print("🖼️ Image Path:", sample['image_path'])
print("🎥 Video:", sample['video_name'])
print("🧠 Frame ID:", sample['frame_id'])

# Pose target
print("\n🎯 SMPL Pose:", sample['pose'].shape)
print("SMPL Pose values:", sample['pose'])

# MediaPipe keypoints
if sample['mediapipe_kpts'] is not None:
    print("\n📌 MediaPipe Keypoints (33, 2):")
    print(sample['mediapipe_kpts'])  # Normalized x, y
else:
    print("\n❌ No MediaPipe keypoints found for this sample.")


🖼️ Image Path: /x1/ljiao/Bhanu/VIBE/output/dance/G_I_DLE_G_I_DLE_LATATA_Lisa_Rhee_Dance_Coverkpop_dance_13_clip1_frames/G_I_DLE_G_I_DLE_LATATA_Lisa_Rhee_Dance_Coverkpop_dance_13_clip1_000064.png
🎥 Video: G_I_DLE_G_I_DLE_LATATA_Lisa_Rhee_Dance_Coverkpop_dance_13_clip1.pkl
🧠 Frame ID: 63

🎯 SMPL Pose: (72,)
SMPL Pose values: [-3.0036819e+00 -1.1432899e-01  2.9013801e-01 -1.2801281e-01
  6.7194767e-02  4.1302481e-01 -3.8798198e-01 -1.3288440e-01
 -3.3363533e-01  3.1728023e-01 -2.3775574e-02  4.5993896e-03
  3.2544896e-01 -1.0814834e-01 -7.6942444e-02  5.1550889e-01
 -8.5658483e-02  3.9466107e-03  4.3211497e-02 -4.6512928e-02
  7.2167255e-04 -2.3313205e-01  1.3888876e-01 -4.1903082e-02
 -2.0071207e-01 -9.1758989e-02  1.2800154e-01 -4.5550045e-02
  2.3208678e-02  5.3244837e-02 -1.7477028e-02  5.0174687e-02
  3.2611089e-03 -3.5222188e-02 -7.9344278e-03 -3.5027906e-03
 -2.3698378e-01  7.4038799e-03 -5.3376608e-02 -6.0079835e-02
 -3.8876846e-01 -2.4517901e-01 -5.0605003e-02  3.9954868e-01
  2.

In [26]:
import torch
from torch.utils.data import Dataset

class SMPLDataset(Dataset):
    def __init__(self, data_list):
        # Filter out samples without keypoints
        self.data = [s for s in data_list if s['mediapipe_kpts'] is not None]

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        sample = self.data[idx]

        # Flatten the 2D keypoints: (33, 2) → (66,)
        x = sample['mediapipe_kpts'].flatten().astype(np.float32)

        # Targets
        pose = sample['pose'].astype(np.float32)    # (72,)
        betas = sample['betas'].astype(np.float32)  # (10,)
        cam = sample['cam'].astype(np.float32)      # (3,)

        y = np.concatenate([pose, betas, cam])      # → (85,)

        return torch.tensor(x), torch.tensor(y)


In [27]:
from torch.utils.data import DataLoader, random_split

# Create full dataset
full_dataset = SMPLDataset(dataset)

# Optional: Split into train/test
train_size = int(0.9 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_set, val_set = random_split(full_dataset, [train_size, val_size])

# DataLoaders
train_loader = DataLoader(train_set, batch_size=64, shuffle=True)
val_loader = DataLoader(val_set, batch_size=64)


In [28]:
import torch.nn as nn

class PoseRegressor(nn.Module):
    def __init__(self, input_dim=66, output_dim=85):
        super(PoseRegressor, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, output_dim)
        )

    def forward(self, x):
        return self.model(x)


In [29]:
import torch
import torch.nn.functional as F
import torch.optim as optim

# Device

device = torch.device("cpu")


# Model
model = PoseRegressor().to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()

# Training
epochs = 20
for epoch in range(epochs):
    model.train()
    train_loss = 0

    for batch_x, batch_y in train_loader:
        batch_x = batch_x.to(device)
        batch_y = batch_y.to(device)

        optimizer.zero_grad()
        preds = model(batch_x)

        loss = loss_fn(preds, batch_y)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    avg_loss = train_loss / len(train_loader)
    print(f"🟢 Epoch {epoch+1}/{epochs} - Train Loss: {avg_loss:.4f}")

    # Validation
    model.eval()
    with torch.no_grad():
        val_loss = 0
        for val_x, val_y in val_loader:
            val_x = val_x.to(device)
            val_y = val_y.to(device)
            preds = model(val_x)
            loss = loss_fn(preds, val_y)
            val_loss += loss.item()

        avg_val_loss = val_loss / len(val_loader)
        print(f"🔵 Validation Loss: {avg_val_loss:.4f}")


🟢 Epoch 1/20 - Train Loss: 0.1204
🔵 Validation Loss: 0.1054
🟢 Epoch 2/20 - Train Loss: 0.1094
🔵 Validation Loss: 0.1071
🟢 Epoch 3/20 - Train Loss: 0.1060
🔵 Validation Loss: 0.0979
🟢 Epoch 4/20 - Train Loss: 0.1029
🔵 Validation Loss: 0.0951
🟢 Epoch 5/20 - Train Loss: 0.1008
🔵 Validation Loss: 0.0964
🟢 Epoch 6/20 - Train Loss: 0.0997
🔵 Validation Loss: 0.0917
🟢 Epoch 7/20 - Train Loss: 0.0981
🔵 Validation Loss: 0.0928
🟢 Epoch 8/20 - Train Loss: 0.0968
🔵 Validation Loss: 0.0947
🟢 Epoch 9/20 - Train Loss: 0.0963
🔵 Validation Loss: 0.0985
🟢 Epoch 10/20 - Train Loss: 0.0949
🔵 Validation Loss: 0.0896
🟢 Epoch 11/20 - Train Loss: 0.0939
🔵 Validation Loss: 0.0857
🟢 Epoch 12/20 - Train Loss: 0.0935
🔵 Validation Loss: 0.0851
🟢 Epoch 13/20 - Train Loss: 0.0930
🔵 Validation Loss: 0.0869
🟢 Epoch 14/20 - Train Loss: 0.0918
🔵 Validation Loss: 0.0874
🟢 Epoch 15/20 - Train Loss: 0.0911
🔵 Validation Loss: 0.0829
🟢 Epoch 16/20 - Train Loss: 0.0909
🔵 Validation Loss: 0.0863
🟢 Epoch 17/20 - Train Loss: 0.090

In [39]:
torch.save(model.state_dict(), "pose_regressor.pth")
print("✅ Model saved as pose_regressor.pth")


✅ Model saved as pose_regressor.pth


In [40]:
# Load model again (if needed)
model = PoseRegressor()
model.load_state_dict(torch.load("pose_regressor.pth", map_location="cpu"))
model.eval()


PoseRegressor(
  (model): Sequential(
    (0): Linear(in_features=66, out_features=128, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.2, inplace=False)
    (3): Linear(in_features=128, out_features=256, bias=True)
    (4): ReLU()
    (5): Dropout(p=0.2, inplace=False)
    (6): Linear(in_features=256, out_features=128, bias=True)
    (7): ReLU()
    (8): Linear(in_features=128, out_features=85, bias=True)
  )
)

In [41]:
import mediapipe as mp
import cv2

def infer_image(image_path):
    # Load and process image
    image = cv2.imread(image_path)
    h, w, _ = image.shape
    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    # Extract 2D keypoints from MediaPipe
    mp_pose = mp.solutions.pose
    with mp_pose.Pose(static_image_mode=True) as pose_detector:
        results = pose_detector.process(image_rgb)

        if not results.pose_landmarks:
            print("❌ No person detected!")
            return None

        # Extract keypoints (normalized)
        kpts = []
        for lm in results.pose_landmarks.landmark:
            kpts.append([lm.x, lm.y])  # Already normalized
        kpts = np.array(kpts).flatten().astype(np.float32)

        # Run inference
        x = torch.tensor(kpts).unsqueeze(0)  # shape (1, 66)
        with torch.no_grad():
            y_pred = model(x).squeeze(0).numpy()

        # Split outputs
        pose_pred = y_pred[:72]
        betas_pred = y_pred[72:82]
        cam_pred = y_pred[82:]

        print("✅ Inference Complete:")
        print("  Pose (72):", pose_pred[:6], "...")
        print("  Betas (10):", betas_pred)
        print("  Cam (3):", cam_pred)

        return {
            "pose": pose_pred,
            "betas": betas_pred,
            "cam": cam_pred,
            "image_path": image_path,
            "keypoints": kpts.reshape(-1, 2)
        }


In [63]:
test_result = infer_image("/x1/ljiao/Bhanu/VIBE/huhu.png")  


✅ Inference Complete:
  Pose (72): [-1.5470067   0.01691272  0.05879398 -0.0811607   0.05268486  0.16186908] ...
  Betas (10): [-0.18334796  1.5337868   0.3686564   0.10613407 -0.4354053  -0.15536767
 -0.19670916  0.29011422  0.40011013  0.14637507]
  Cam (3): [ 1.0967458  -0.03771771  0.11843612]


I0000 00:00:1743848763.129702 3032688 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1743848763.150833 3382461 gl_context.cc:369] GL version: 3.2 (OpenGL ES 3.2 NVIDIA 535.183.01), renderer: NVIDIA A100 80GB PCIe/PCIe/SSE2
W0000 00:00:1743848763.204910 3382390 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1743848763.257007 3382416 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


In [42]:
pip install smplx trimesh pyrender


Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [64]:
SMPL_MODEL_PATH = "/x1/ljiao/Bhanu/VIBE/SMPL_MALE.pkl"


In [44]:
!pip uninstall numpy -y
!pip install numpy==1.23.5


Found existing installation: numpy 1.23.5
Uninstalling numpy-1.23.5:
  Successfully uninstalled numpy-1.23.5
Defaulting to user installation because normal site-packages is not writeable
  Using cached numpy-1.23.5-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (17.1 MB)
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
mediapipe 0.10.21 requires protobuf<5,>=4.25.3, but you have protobuf 5.29.3 which is incompatible.
k-means-constrained 0.7.5 requires numpy>=2.1.1, but you have numpy 1.23.5 which is incompatible.
jaxlib 0.5.1 requires numpy>=1.25, but you have numpy 1.23.5 which is incompatible.
jax 0.5.1 requires numpy>=1.25, but you have numpy 1.23.5 which is incompatible.


In [45]:
import numpy as np
print(np.__version__)  # Should say 1.23.5


1.23.5


In [60]:
import smplx
import trimesh
import torch
import pyrender
import numpy as np

def visualize_smpl(prediction, smpl_model_path):
    import os
    import smplx
    import torch
    import trimesh
    import pyrender

    pose = torch.tensor(prediction['pose']).unsqueeze(0).float()
    betas = torch.tensor(prediction['betas']).unsqueeze(0).float()
    cam = torch.tensor(prediction['cam']).unsqueeze(0).float()

    # ✅ Fix: Pass full file path instead of just directory
    model = smplx.SMPL(
        model_path=smpl_model_path,
        gender='male',  # or 'neutral' if that's what you have
        batch_size=1
    )

    # Get mesh
    output = model(body_pose=pose[:, 3:], global_orient=pose[:, :3], betas=betas, pose2rot=True)
    vertices = output.vertices.detach().cpu().numpy().squeeze()
    faces = model.faces

    # Visualize
    mesh = trimesh.Trimesh(vertices, faces, process=False)
    scene = pyrender.Scene()
    mesh_node = pyrender.Mesh.from_trimesh(mesh)
    scene.add(mesh_node)
    pyrender.Viewer(scene, use_raymond_lighting=True)

visualize_smpl(
    test_result,
    smpl_model_path="/x1/ljiao/Bhanu/VIBE/SMPL_MALE.pkl"  # correct full path
)



NoSuchDisplayException: Cannot connect to "None"

In [ ]:
visualize_smpl(
    test_result,
    smpl_model_path="/x1/ljiao/Bhanu/VIBE/SMPL_MALE.pkl"  # correct full path
)



In [65]:
def save_mesh_as_obj(prediction, smpl_model_path, save_path="predicted_mesh.obj"):
    pose = torch.tensor(prediction['pose']).unsqueeze(0).float()
    betas = torch.tensor(prediction['betas']).unsqueeze(0).float()

    model = smplx.SMPL(
        model_path=smpl_model_path,
        gender='male',  # or 'neutral'
        batch_size=1
    )

    output = model(
        body_pose=pose[:, 3:],
        global_orient=pose[:, :3],
        betas=betas,
        pose2rot=True
    )
    vertices = output.vertices.detach().cpu().numpy().squeeze()
    faces = model.faces

    mesh = trimesh.Trimesh(vertices, faces, process=False)
    mesh.export(save_path)
    print(f"✅ 3D Mesh saved as: {save_path}")




In [66]:
save_mesh_as_obj(
    prediction=test_result,
    smpl_model_path="/x1/ljiao/Bhanu/VIBE/SMPL_MALE.pkl",
    save_path="predicted_mesh.obj"
)




✅ 3D Mesh saved as: predicted_mesh.obj


In [54]:
sudo apt-get install libosmesa6-dev libgl1-mesa-glx libglfw3



SyntaxError: invalid syntax (2858393344.py, line 1)